In [71]:
import pandas as pd

# 1) Archivo de grupos (alimentos / no alimentos / general)
grupos = pd.read_csv(r"C:\Users\ASUS\Desktop\CFBPredic\data\ipc.csv")
# 2) Archivo de detalle por producto
productos = pd.read_csv(r"C:\Users\ASUS\Desktop\CFBPredic\data\productos.csv")


In [72]:
grupos.head()

,Año,Mes,Grupo,Subgrupo,Indicador_grup,Indicador
0,2006,Enero,Alimentos y No alimentos,Alimentos,60.91,Índice
1,2006,Febrero,Alimentos y No alimentos,Alimentos,62.03,Índice
2,2006,Febrero,Alimentos y No alimentos,Alimentos,1.84,Variación mensual
3,2006,Marzo,Alimentos y No alimentos,Alimentos,63.10,Índice
4,2006,Marzo,Alimentos y No alimentos,Alimentos,1.73,Variación mensual


In [73]:
productos.head()

,Año,Mes,'Series_IPC'[Ciudad],Nivel,Cód. CCIF,Descripción CCIF,'Indicadores_cuboIPC'[Indicador],'Filtro Indicador'[Indicador]
0,2005,Enero,Nacional,Clase,111,Pan y cereales (ND),51.48,Índice
1,2005,Enero,Nacional,Clase,112,Carne (ND),56.89,Índice
2,2005,Enero,Nacional,Clase,113,Pescado (ND),55.78,Índice
3,2005,Enero,Nacional,Clase,114,"Leche, queso y huevos (ND)",62.85,Índice
4,2005,Enero,Nacional,Clase,115,Aceites y grasas (ND),51.42,Índice


In [74]:
# Mapeo de meses en español a número
map_mes = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Setiembre": 9,  # por si acaso
    "Octubre": 10, "Noviembre": 11, "Diciembre": 12
}

for df in [grupos, productos]:
    df["mes_num"] = df["Mes"].map(map_mes)
    df["date"] = pd.to_datetime(
        dict(year=df["Año"], month=df["mes_num"], day=1)
    )


In [75]:
grupos.head()

,Año,Mes,Grupo,Subgrupo,Indicador_grup,Indicador,mes_num,date
0,2006,Enero,Alimentos y No alimentos,Alimentos,60.91,Índice,1,2006-01-01
1,2006,Febrero,Alimentos y No alimentos,Alimentos,62.03,Índice,2,2006-02-01
2,2006,Febrero,Alimentos y No alimentos,Alimentos,1.84,Variación mensual,2,2006-02-01
3,2006,Marzo,Alimentos y No alimentos,Alimentos,63.10,Índice,3,2006-03-01
4,2006,Marzo,Alimentos y No alimentos,Alimentos,1.73,Variación mensual,3,2006-03-01


In [76]:
alimentos = grupos[
    (grupos["Indicador"] == "Índice")
].copy()

alimentos = alimentos[["date", "Indicador_grup"]].rename(
    columns={"Indicador_grup": "ipc_alimentos_index"}
).sort_values("date").reset_index(drop=True)

In [77]:
alimentos

,date,ipc_alimentos_index
0,2006-01-01,60.910000
1,2006-02-01,62.030000
2,2006-03-01,63.100000
3,2006-04-01,62.450000
4,2006-05-01,61.910000
...,...,...
233,2025-06-01,120.425104
234,2025-07-01,120.982157
235,2025-08-01,121.515398
236,2025-09-01,121.422272


In [78]:
productos = productos[
    (productos["'Filtro Indicador'[Indicador]"] == "Índice") &
    (productos["Nivel"] == "Clase")
].copy()
productos = productos.rename(columns={
    "Año": "year",
    "Mes": "month",
    "'Series_IPC'[Ciudad]": "ciudad",
    "Nivel": "nivel",
    "Cód. CCIF": "ccif",
    "Descripción CCIF": "descripcion",
    "'Indicadores_cuboIPC'[Indicador]": "valor",
    "'Filtro Indicador'[Indicador]": "tipo_indicador"
})

productos = productos[["date", "descripcion", "valor"]].copy()

In [79]:
productos.head()

,date,descripcion,valor
0,2005-01-01,Pan y cereales (ND),51.48
1,2005-01-01,Carne (ND),56.89
2,2005-01-01,Pescado (ND),55.78
3,2005-01-01,"Leche, queso y huevos (ND)",62.85
4,2005-01-01,Aceites y grasas (ND),51.42


In [80]:
panel_productos = productos.pivot_table(
    index="date",
    columns="descripcion",
    values="valor"
).reset_index()


In [81]:
panel_productos.columns = [
    col.lower()
       .replace(" ", "_")
       .replace("-", "_")
       .replace("(nd)", "")
       .replace(",", "")
       .replace('"', '')
       .replace("á", "a")
       .replace("é", "e")
       .replace("í", "i")
       .replace("ó", "o")
       .replace("ú", "u")
       .replace("ñ", "n")
    for col in panel_productos.columns
]


In [82]:
data = alimentos.merge(panel_productos, on="date", how="inner").sort_values("date")
data.columns = [col.rstrip("_") for col in data.columns]


In [83]:
data.head()

,date,ipc_alimentos_index,aceites_y_grasas,aguas_minerales_refrescos_jugos_de_frutas_y_de_legumbres,azucar_mermelada_miel_chocolate_y_dulces_de_azucar,cafe_te_y_cacao,carne,frutas,leche_queso_y_huevos,legumbres_hortalizas,pan_y_cereales,pescado,productos_alimenticios_n.e.p.
0,2006-01-01,60.91,51.33,62.47,55.45,48.95,61.03,63.57,66.73,72.88,50.38,57.44,60.35
1,2006-02-01,62.03,50.93,63.41,56.86,48.23,63.90,66.18,66.12,74.62,50.30,58.12,59.94
2,2006-03-01,63.10,51.28,63.43,58.43,49.08,64.41,69.87,66.05,77.43,50.69,59.30,60.96
3,2006-04-01,62.45,51.24,62.47,60.56,49.07,62.82,67.99,66.55,75.19,50.32,61.94,61.56
4,2006-05-01,61.91,51.36,62.40,64.10,49.06,62.92,66.12,67.07,70.09,50.52,63.21,61.78


In [ ]:
# data.to_csv(r'C:\Users\ASUS\Desktop\CFBPredic\src\data.csv', 
#                    index=False)